In [ ]:
# %pip install pytorch-lightning
from torchvision import transforms as T
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
import pandas as pd
from tqdm import tqdm
import numpy as np
import pytorch_lightning as L
from pytorch_lightning.strategies import DDPStrategy
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.utilities import CombinedLoader
from torch.cuda.amp import autocast, GradScaler
import os
import random
import torch.nn.functional as F
from collections import deque
from PIL import Image

# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d arnaud58/selfie2anime
# !unzip selfie2anime.zip
# !kaggle competitions download -c gan-getting-started
# !unzip gan-getting-started.zip
# os.listdir("/kaggle/input")
# pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
class TRANSFORMS:
    def __init__(self, l_dim=256, t_dim=256):
        self.transform_train = T.Compose([
            T.Resize((l_dim, l_dim), antialias=True),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])
        self.transform = T.Compose([
            T.Resize((l_dim, l_dim), antialias=True),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

    def __call__(self, image, stage):
        if stage == 'fit':
            image = self.transform_train(image)
        else:
            image = self.transform(image)
        return image

In [ ]:
# photo_path = "/content/testB/"
# photo_path = "/content/trainB/"
# photo_path = "/kaggle/input/selfie2anime/trainA/"
photo_path = "/kaggle/input/selfie2anime/trainB/"
batch_size = 16

class MDataset(torch.utils.data.Dataset):
    def __init__(self, filenames, path, transforms, stage):
        self.filenames = filenames
        self.path = path
        self.transforms = transforms
        self.stage = stage
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        name = self.path + self.filenames[idx]
        image = Image.open(name)
        return self.transforms(image, self.stage)

In [ ]:
class Module(L.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.photo_names = os.listdir(photo_path)
        self.transform = TRANSFORMS()
    def setup(self, stage):
        if stage == "fit":
            self.train_photo = MDataset(self.photo_names, photo_path, self.transform, "fit")
    def train_dataloader(self):
        photo_loader = DataLoader(self.train_photo, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=3)
        return photo_loader
    def val_dataloader(self):
        photo_loader = DataLoader(self.train_photo, batch_size=6, shuffle=True, drop_last=True, num_workers=1)
        return photo_loader

In [ ]:
def show_images(tensor_images, n=5):
    images = tensor_images[:n].detach().cpu() / 2 + 0.5
    print(images.shape)
    plt.figure(figsize=(15, 3))
    for i in range(n):
        img = T.functional .to_pil_image(images[i])
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
def compute_gp(D, real, fake):
  alpha = torch.rand(real.size(0), 1, 1, 1, device=real.device)
  inter = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
  critic_inter = D(inter)
  gradients = torch.autograd.grad(
      inputs=inter,
      outputs=critic_inter,
      grad_outputs=torch.ones_like(critic_inter),
      create_graph=True,
      retain_graph=True,
  )[0]
  gradients = gradients.view(gradients.size(0), -1)
  gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
  return gp

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon
    def forward(self, x):
        return x * torch.rsqrt(torch.mean(x.pow(2.0), dim=1, keepdim=True) + self.epsilon)
        
class Upscale(nn.Module):
    def __init__(self, factor=2, gain=1):
        super().__init__()
        self.factor = factor
        self.gain = gain
    def forward(self, x):
        size = x.shape
        x = x.view(size[0], size[1], size[2], 1, size[3], 1).expand(-1, -1, -1, self.factor, -1, self.factor)
        x = x.contiguous().view(size[0], size[1], size[2] * self.factor, size[3] * self.factor)
        return x
        
class ToRGB(nn.Module):
    def __init__(self, inputs, channels=3):
        super().__init__()
        self.inputs = inputs
        self.channels = channels
        self.to_rgb = nn.Conv2d(inputs, channels, kernel_size=1)
    def forward(self, x):
        x = self.to_rgb(x)
        return x
        
class FromRGB(nn.Module):
    def __init__(self, inputs, outputs):
        super().__init__()
        self.inputs = inputs
        self.channels = outputs
        self.from_rgb = nn.Conv2d(inputs, outputs, kernel_size=1)
    def forward(self, x):
        x = self.from_rgb(x)
        return x
        
class MinibatchStdDev(nn.Module):
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon
    def forward(self, x):
        N, C, H, W = x.shape
        std = torch.std(x, dim=0, unbiased=False) + self.epsilon
        std = std.mean(dim=0, keepdim=True)
        std = std.expand(N, 1, H, W)
        return torch.cat([x, std], dim=1)
        
class AdaIN(nn.Module):
    def __init__(self, latent_size, num_features):
        super().__init__()
        self.fc = nn.Linear(latent_size, num_features * 2)

    def forward(self, w, x):
        B, C = x.shape[:2]
        A = self.fc(w)
        x_mean = x.mean(dim=[2, 3], keepdim=True)
        x_std = x.std(dim=[2, 3], keepdim=True)
        normilized = (x - x_mean) / (x_std + 1e-8)

        gamma, beta = A.chunk(2, dim=1)
        gamma = gamma.view(B, C, 1, 1)
        beta = beta.view(B, C, 1, 1)

        return normilized * gamma + beta
        
class Noise(nn.Module):
    def __init__(self, channels):
      super().__init__()
      self.scale = nn.Parameter(torch.zeros(channels))

    def forward(self, x):
      noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
      return x + noise * self.scale.view(1, -1, 1, 1)
        
class Blur(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        f = np.array([1, 2, 1], dtype=np.float32)
        f = np.outer(f, f) 
        f /= f.sum()     

        kernel = torch.tensor(f, dtype=torch.float32).view(1, 1, 3, 3)
        kernel = kernel.repeat(channels, 1, 1, 1)
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False, groups=channels)
        with torch.no_grad():
            self.conv.weight.copy_(kernel)

        for param in self.conv.parameters():
            param.requires_grad = False
    def forward(self, x):
        return self.conv(x)

class Maping(nn.Module):
    def __init__(self, latent_size=512, layers=8):
        super().__init__()
        self.layers = layers
        layers_list = nn.ModuleList()
        layers_list.append(PixelNorm())
        for _ in range(layers):
            layers_list.append(nn.Linear(latent_size, latent_size))
            layers_list.append(nn.LeakyReLU(0.2))
        layers_list.append(PixelNorm())
        self.net = nn.Sequential(*layers_list)
    def forward(self, x):
        return self.net(x)
        
class DiscrBlock(nn.Module):
  def __init__(self, in_channels, out_channels, last=True):
    super().__init__()
    self.last = last
    if last:
        self.block = nn.Sequential(
            MinibatchStdDev(),
            nn.Conv2d(in_channels + 1, in_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(in_channels * 4 * 4, out_channels),
            nn.LeakyReLU(0.2),
        )
    else:
      self.block = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
          nn.LeakyReLU(0.2),
          nn.InstanceNorm2d(out_channels),
          nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
          nn.AvgPool2d(2),
          Blur(out_channels),
          nn.LeakyReLU(0.2),
          nn.InstanceNorm2d(out_channels),
      )
  def forward(self, x):
    return self.block(x)
      
class Synthesis(nn.Module):
  def __init__(self, in_channels, out_channels, has_conv=True, latent_size=512):
    super().__init__()
    self.has_conv = has_conv
    self.up =  Upscale()
    self.conv_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=True)
    self.noise_1 = Noise(out_channels)
    self.act_1 = nn.LeakyReLU(0.2)
    self.adain_1 = AdaIN(latent_size, out_channels)
    self.conv_2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=True)
    self.noise_2 = Noise(out_channels)
    self.act_2 = nn.LeakyReLU(0.2)
    self.adain_2 = AdaIN(latent_size, out_channels)

  def forward(self, w, x):
    if self.has_conv:
      x = self.up(x)
    x = self.conv_1(x)
    x = self.noise_1(x)
    x = self.act_1(x)
    x = self.adain_1(w, x)
    x = self.conv_2(x)
    x = self.noise_2(x)
    x = self.act_2(x)
    x = self.adain_2(w, x)

    return x

In [ ]:
# @title model
epochs = 50
latent_size = 256

class Generator(nn.Module):
    def __init__(self, startf=32, maxf=256, layers=7, latent_size=256):
        super().__init__()
        self.state = nn.Parameter(torch.randn(1, maxf, 4, 4))
        self.style = Maping(latent_size)
        self.synthesis = nn.ModuleList()
        self.to_rgb = nn.ModuleList()
        mul = 2 ** (layers-1)
        inputs = min(maxf, startf * mul)

        for i in range(layers):
          outputs = min(maxf, startf * mul)
          has_conv = i != 0
          block = Synthesis(inputs, outputs, has_conv, latent_size)
          self.synthesis.append(block)
          inputs = outputs
          self.to_rgb.append(ToRGB(outputs, 3))
          mul //= 2

    def forward(self, z, lod=6):
        style = self.style(z)
        x = self.state.repeat(z.size(0), 1, 1, 1)
        for i in range(lod+1):
            x = self.synthesis[i](style, x)
        x = self.to_rgb[lod](x)
        return x

In [ ]:
class Discriminator(nn.Module):
  def __init__(self, startf=32, maxf=256, layers=7):
    super().__init__()
    self.from_rgb = nn.ModuleList()
    self.net = nn.ModuleList()
    self.layers = layers
    mul = 2
    inputs = startf
      
    for i in range(layers):
        outputs = min(maxf, startf * mul)
        last = i+1 == layers
        block = DiscrBlock(inputs, outputs, last)
        self.net.append(block)
        self.from_rgb.append(FromRGB(3, inputs))
        inputs = outputs
        mul *= 2
    self.fc = nn.Linear(inputs, 1)
  def forward(self, x, lod=6):
    x = self.from_rgb[self.layers - lod - 1](x)
    for i in range(self.layers - lod - 1, self.layers):
        x = self.net[i](x)
    x = self.fc(x)
    return x

In [ ]:
class GAN(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.G = Generator(latent_size=latent_size)
        self.D = Discriminator()
        self.automatic_optimization = False

    def forward(self, z):
        return self.G(z)

    def training_step(self, batch, batch_idx):
        G_optim, D_optim = self.optimizers()

        for _ in range(1):
            z = torch.randn(batch_size, latent_size, device=self.device)
            with torch.no_grad():
                photo_gen = self.G(z)

            real_predicted = self.D(batch)
            fake_predicted = self.D(photo_gen)
            gradient_penalty = compute_gp(self.D, batch, photo_gen)

            D_loss = -real_predicted.mean() + fake_predicted.mean() + gradient_penalty * 10

            D_optim.zero_grad()
            self.manual_backward(D_loss)
            D_optim.step()

        photo_gen = self.G(z)
        fake_predicted = self.D(photo_gen)

        G_loss = -fake_predicted.mean()
        G_optim.zero_grad()
        self.manual_backward(G_loss)
        G_optim.step()

        self.log("loss_D", D_loss, prog_bar=True, on_epoch=True)
        self.log("loss_G", G_loss, prog_bar=True, on_epoch=True)

        return G_loss + D_loss

    def validation_step(self, batch, batch_idx):
        z = torch.randn(6, latent_size, device=self.device)
        images = self.G(z)
        show_images(images, n=6)

        loss_D = self.trainer.callback_metrics.get("loss_D")
        loss_G = self.trainer.callback_metrics.get("loss_G")

        if loss_D is not None and loss_G is not None:
            print(f"[Epoch {self.current_epoch}] Avg D Loss: {loss_D.item():.4f} | Avg G Loss: {loss_G.item():.4f}")

    def configure_optimizers(self):
        g_params = self.G.parameters()
        d_params = self.D.parameters()

        opt_g = torch.optim.Adam(g_params, lr=2e-4, betas=(0.0, 0.999))
        opt_d = torch.optim.Adam(d_params, lr=2e-4, betas=(0.0, 0.999))
        return [opt_g, opt_d], []

In [ ]:
model = GAN()
loader = Module()
trainer = L.Trainer(max_epochs=50,
                    devices=2,
                    num_sanity_val_steps=1,
                    limit_val_batches=1,
                    # precision="16-mixed",
                    accelerator="gpu",
                    strategy="ddp_notebook_find_unused_parameters_true",
                   )
trainer.fit(model, loader,)